## Example 1: grpcio_status functionality test 1

In [ ]:
from grpc_status import rpc_status
from google.rpc import status_pb2
from google.rpc import code_pb2
from google.protobuf import any_pb2
import grpc

# Example 1: Basic gRPC status code mapping and error handling
print("Example 1: Basic gRPC status code mapping and error handling")

try:
    # Explore available status codes
    status_codes = [attr for attr in dir(code_pb2) if not attr.startswith('_') and attr.isupper()]
    print(f"✓ Available gRPC status codes: {len(status_codes)}")
    print(f"  Common codes: {status_codes[:5]}")
    
    # Create a status proto message
    status = status_pb2.Status()
    status.code = code_pb2.INVALID_ARGUMENT
    status.message = "Invalid request parameter"
    
    print(f"\n✓ Status proto created:")
    print(f"  Code: {status.code} ({code_pb2.Code.Name(status.code)})")
    print(f"  Message: {status.message}")
    
    # Test code to gRPC status mapping
    grpc_code = rpc_status.code_to_grpc_status_code(status.code)
    print(f"  Mapped gRPC code: {grpc_code}")
    
    # Create status with details
    status.details.extend([
        # API drift: `Any` lives in `google.protobuf.any_pb2`. `google.rpc.status_pb2` only ever
        # re-exported it as an import side effect, and it now exposes `Status` alone.
        any_pb2.Any(type_url="type.googleapis.com/google.protobuf.StringValue",
                    value=b"Additional error context")
    ])
    print(f"  Status details: {len(status.details)} item(s)")
    
    print("✓ Basic status mapping completed")
    
except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"grpcio_status example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: grpcio_status functionality test 2

In [ ]:
from grpc_status import rpc_status
from google.rpc import status_pb2
from google.rpc import code_pb2
import grpc

# Example 2: RPC error status creation and conversion
print("Example 2: RPC error status creation and conversion")

try:
    # Common gRPC error scenarios
    error_scenarios = [
        (code_pb2.NOT_FOUND, "Resource not found"),
        (code_pb2.PERMISSION_DENIED, "Access denied"), 
        (code_pb2.RESOURCE_EXHAUSTED, "Rate limit exceeded"),
        (code_pb2.DEADLINE_EXCEEDED, "Request timeout"),
        (code_pb2.ALREADY_EXISTS, "Resource already exists")
    ]
    
    print("✓ Creating status objects for common errors:")
    
    statuses = []
    for code, message in error_scenarios:
        status = status_pb2.Status()
        status.code = code
        status.message = message
        
        # Map to gRPC status code
        grpc_code = rpc_status.code_to_grpc_status_code(code)
        code_name = code_pb2.Code.Name(code)
        
        print(f"  {code_name}: {message}")
        print(f"    gRPC code: {grpc_code}")
        
        statuses.append(status)
    
    # Demonstrate status serialization
    sample_status = statuses[0]
    serialized = sample_status.SerializeToString()
    print(f"\n✓ Status serialization:")
    print(f"  Serialized size: {len(serialized)} bytes")
    
    # Parse back
    parsed_status = status_pb2.Status()
    parsed_status.ParseFromString(serialized)
    print(f"  Parsed code: {code_pb2.Code.Name(parsed_status.code)}")
    print(f"  Parsed message: {parsed_status.message}")
    
    print("✓ RPC error status conversion completed")
    
except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"grpcio_status example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: grpcio_status functionality test 3

In [ ]:
from grpc_status import rpc_status
from google.rpc import status_pb2
from google.rpc import code_pb2
from google.protobuf import any_pb2
import grpc

# Example 3: Status details and rich error information
print("Example 3: Status details and rich error information")

try:
    # Create status with detailed error information
    status = status_pb2.Status()
    status.code = code_pb2.FAILED_PRECONDITION
    status.message = "Operation failed due to precondition"
    
    # Add structured error details
    error_details = [
        {
            "type_url": "type.googleapis.com/google.rpc.BadRequest",
            "field": "user_id",
            "description": "Invalid user identifier format"
        },
        {
            "type_url": "type.googleapis.com/google.rpc.PreconditionFailure", 
            "subject": "account_status",
            "description": "Account must be verified before operation"
        }
    ]
    
    print(f"✓ Status with rich error details:")
    print(f"  Code: {code_pb2.Code.Name(status.code)}")
    print(f"  Message: {status.message}")
    
    # Pack error details as Any messages
    for detail in error_details:
        any_detail = any_pb2.Any()
        any_detail.type_url = detail["type_url"] 
        any_detail.value = f"{detail.get('field', detail.get('subject', 'error'))}: {detail['description']}".encode()
        
        status.details.append(any_detail)
        print(f"    Detail: {detail['type_url'].split('/')[-1]}")
    
    print(f"  Total details: {len(status.details)}")
    
    # Demonstrate metadata key for status details
    metadata_key = rpc_status.GRPC_DETAILS_METADATA_KEY
    print(f"\n✓ gRPC metadata key: {metadata_key}")
    
    # Status code mapping verification
    mapped_code = rpc_status.code_to_grpc_status_code(status.code)
    print(f"✓ Mapped to gRPC code: {mapped_code}")
    
    print("✓ Rich error information handling completed")
    
except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"grpcio_status example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: grpcio_status functionality test 4

In [ ]:
from grpc_status import rpc_status
from google.rpc import status_pb2
from google.rpc import code_pb2
import grpc

# Example 4: Status extraction from gRPC calls and error handling
print("Example 4: Status extraction from gRPC calls and error handling")

try:
    # Mock gRPC call context for demonstration
    class MockCall:
        def __init__(self, code, details_metadata=None):
            self.code = code
            self.details_metadata = details_metadata or {}
            
        def code(self):
            return self.code
            
        def trailing_metadata(self):
            return [(key, value) for key, value in self.details_metadata.items()]
    
    # Create mock calls with different status scenarios
    mock_calls = [
        (MockCall(grpc.StatusCode.OK), "Success case"),
        (MockCall(grpc.StatusCode.NOT_FOUND), "Resource not found"),
        (MockCall(grpc.StatusCode.PERMISSION_DENIED), "Access denied"),
        (MockCall(grpc.StatusCode.INTERNAL), "Internal server error")
    ]
    
    print("✓ Processing mock gRPC calls:")
    
    for mock_call, description in mock_calls:
        try:
            # Attempt to extract status from call
            # Note: from_call might not work with mock objects, so we demonstrate the concept
            print(f"  {description}:")
            print(f"    Call status: {mock_call.code}")
            
            # Show what the real from_call would extract
            if hasattr(rpc_status, 'from_call'):
                print(f"    from_call function available for status extraction")
            
        except Exception as call_error:
            print(f"    Call processing: {str(call_error)[:50]}...")
            raise
    
    # Demonstrate status code enum values
    status_mappings = [
        (grpc.StatusCode.OK, code_pb2.OK),
        (grpc.StatusCode.INVALID_ARGUMENT, code_pb2.INVALID_ARGUMENT), 
        (grpc.StatusCode.NOT_FOUND, code_pb2.NOT_FOUND),
        (grpc.StatusCode.PERMISSION_DENIED, code_pb2.PERMISSION_DENIED)
    ]
    
    print(f"\n✓ Status code mappings:")
    for grpc_code, rpc_code in status_mappings:
        print(f"  {grpc_code} -> {code_pb2.Code.Name(rpc_code)}")
    
    print("✓ Status extraction demonstration completed")
    
except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"grpcio_status example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: grpcio_status functionality test 5

In [ ]:
from grpc_status import rpc_status
from google.rpc import status_pb2
from google.rpc import code_pb2
from google.protobuf import any_pb2
import grpc
import json

# Example 5: Advanced status handling and error reporting patterns
print("Example 5: Advanced status handling and error reporting patterns")

try:
    # Complete error status with comprehensive details
    def create_detailed_error(code, message, context=None):
        status = status_pb2.Status()
        status.code = code
        status.message = message
        
        if context:
            for detail_type, detail_data in context.items():
                any_detail = any_pb2.Any()
                any_detail.type_url = f"type.googleapis.com/google.rpc.{detail_type}"
                any_detail.value = json.dumps(detail_data).encode('utf-8')
                status.details.append(any_detail)
        
        return status
    
    # Create complex error scenarios
    error_contexts = {
        "ValidationError": {
            "BadRequest": {
                "field_violations": [
                    {"field": "email", "description": "Invalid email format"},
                    {"field": "age", "description": "Must be between 18 and 120"}
                ]
            }
        },
        "AuthorizationError": {
            "ErrorInfo": {
                "reason": "INSUFFICIENT_PERMISSIONS",
                "domain": "user.service.com",
                "metadata": {"required_role": "admin", "current_role": "user"}
            }
        },
        "ResourceError": {
            "ResourceInfo": {
                "resource_type": "User",
                "resource_name": "users/12345",
                "owner": "user.service.com"
            }
        }
    }
    
    print("✓ Creating detailed error statuses:")
    
    statuses = []
    for error_name, context in error_contexts.items():
        if error_name == "ValidationError":
            code = code_pb2.INVALID_ARGUMENT
        elif error_name == "AuthorizationError": 
            code = code_pb2.PERMISSION_DENIED
        else:
            code = code_pb2.NOT_FOUND
            
        status = create_detailed_error(code, f"{error_name} occurred", context)
        statuses.append(status)
        
        print(f"  {error_name}:")
        print(f"    Code: {code_pb2.Code.Name(status.code)}")
        print(f"    Details: {len(status.details)} item(s)")
        print(f"    gRPC mapping: {rpc_status.code_to_grpc_status_code(status.code)}")
    
    # Demonstrate complete error handling workflow
    print(f"\n✓ Error handling workflow summary:")
    print(f"  Total error patterns: {len(statuses)}")
    print(f"  Metadata key for details: {rpc_status.GRPC_DETAILS_METADATA_KEY}")
    
    print("✓ Advanced status handling patterns completed")
    
except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"grpcio_status example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e

In [ ]:
# Test various import patterns for grpcio-status
print("Testing grpcio-status imports:")

# Each handler re-raises: a broken import must fail this cell, not print "✗ … import failed" and pass.
try:
    import grpc
    print("✓ grpc imported successfully")
except ImportError as e:
    print(f"✗ grpc import failed: {e}")
    raise

try:
    from grpc_status import rpc_status
    print("✓ grpc_status.rpc_status imported successfully")
    rpc_attrs = [attr for attr in dir(rpc_status) if not attr.startswith('_')]
    print(f"  Available attributes: {rpc_attrs[:5]}")
    assert {"from_call", "to_status", "GRPC_DETAILS_METADATA_KEY"} <= set(rpc_attrs), rpc_attrs
except ImportError as e:
    print(f"✗ grpc_status.rpc_status import failed: {e}")
    raise

try:
    from google.rpc import status_pb2
    print("✓ google.rpc.status_pb2 imported successfully")
    assert status_pb2.Status.DESCRIPTOR.full_name == "google.rpc.Status"
except ImportError as e:
    print(f"✗ google.rpc.status_pb2 import failed: {e}")
    raise

try:
    from google.rpc import code_pb2
    print("✓ google.rpc.code_pb2 imported successfully") 
    codes = [attr for attr in dir(code_pb2) if not attr.startswith('_') and 'Code' not in attr]
    print(f"  Available codes: {codes[:5]}")
    assert code_pb2.OK == 0 and code_pb2.NOT_FOUND == 5, (code_pb2.OK, code_pb2.NOT_FOUND)
except ImportError as e:
    print(f"✗ google.rpc.code_pb2 import failed: {e}")
    raise

try:
    from grpc_status.rpc_status import from_call, to_status
    print("✓ from_call and to_status imported successfully")
    assert from_call is rpc_status.from_call and to_status is rpc_status.to_status
except ImportError as e:
    print(f"✗ from_call/to_status import failed: {e}")
    raise

print("\nPackage structure exploration complete")